# Phase 4: Nucleotide Transformer fine-tuning
Run with a Colab GPU. The Phase 2 dataset must be uploaded to `MyDrive/variantfx/data/labeled_split_dataset.tsv`; checkpoints, reports, and MLflow runs persist under `MyDrive/variantfx/phase4`. The notebook contains orchestration only; model logic remains in `src/models/finetune_lm.py`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

Mounted at /content/drive
Mon Aug 17 01:02:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [2]:
!pip install -q transformers==4.42.3 tokenizers==0.19.1 huggingface-hub==0.23.4 mlflow==2.14.1 pandas==2.2.2 scikit-learn==1.5.0
!git clone https://github.com/djasleen15/genomic-variant-prioritizer.git /content/genomic-variant-prioritizer || git -C /content/genomic-variant-prioritizer pull
%cd /content/genomic-variant-prioritizer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.6/402.6 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [ ]:
from pathlib import Path
dataset = Path('/content/drive/MyDrive/variantfx/data/labeled_split_dataset.tsv')
output = Path('/content/drive/MyDrive/variantfx/phase4/artifacts')
mlruns = Path('/content/drive/MyDrive/variantfx/phase4/mlruns')
assert dataset.exists(), f'Upload the Phase 2 dataset to {dataset}'
output.mkdir(parents=True, exist_ok=True)
mlruns.mkdir(parents=True, exist_ok=True)

Start with full fine-tuning. If it runs out of GPU memory or cannot complete within the Colab session, rerun with `--freeze-encoder` as the documented compute fallback. Reduce `--batch-size` before freezing if the failure is memory-only.

In [ ]:
!python -m src.models.finetune_lm --input "{dataset}" --output-dir "{output}" --mlflow-dir "{mlruns}" --batch-size 16 --epochs 3 --patience 1 --audit-samples 100

In [ ]:
import json
report = json.loads((output / 'phase4_report.json').read_text())
print(json.dumps({k: report[k] for k in ['training_approach', 'validation', 'test', 'baseline', 'improvement', 'mlflow_run_id']}, indent=2))